## Spatial sensitivity analysis for utility metrics that leverages spatial discretization
### The execution takes some time as we iterate through 20 cell edge lengths and 5 offsets (20*5)

In [ ]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import sys
import psutil

import matplotlib.pyplot as plt

import pandas as pd

import numpy as np
from tqdm import tqdm
import argparse

from syntraj_utileval.datasets.dataset import TrajectoryDataset

from syntraj_utileval.suite.suite import (
    EvaluationSuite,
    EvalParams,
)
import pandas as pd

from syntraj_utileval.stats.spatial import make_density_grid_from_df

from joblib import Parallel, delayed
from scipy.stats import gaussian_kde, kendalltau, wasserstein_distance


def uniform_datetime_df(df, hour_col=None, day_col=None):
    if hour_col is not None and day_col is not None:
        df["datetime"] = (
            pd.to_datetime("2024-01-01")
            + pd.to_timedelta(df[day_col], unit="D")
            + pd.to_timedelta(df[hour_col], unit="h")
        )
        df = df.drop(columns=[hour_col, day_col])
        return df


In [2]:
df_original_FS = pd.read_csv("../data/weeklyFS-NYC/test_latlon.csv")
df_original_FS = uniform_datetime_df(df_original_FS, hour_col="hour", day_col="day")
orig_FS = TrajectoryDataset(
    df=df_original_FS,
    dataset_name="original",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    datetime_col="datetime",
    semantic_category_col="category",
)

### Case A

In [ ]:
import numpy as np
from tqdm import tqdm

from syntraj_utileval.datasets.dataset import TrajectoryDataset

from syntraj_utileval.suite.suite import (
    EvaluationSuite,
    EvalParams,
)
import pandas as pd
from joblib import Parallel, delayed


def uniform_datetime_df(df, hour_col=None, day_col=None):
    if hour_col is not None and day_col is not None:
        df["datetime"] = (
            pd.to_datetime("2024-01-01")
            + pd.to_timedelta(df[day_col], unit="D")
            + pd.to_timedelta(df[hour_col], unit="h")
        )
        df = df.drop(columns=[hour_col, day_col])
        return df


df_original_FS = pd.read_csv("../data/weeklyFS-NYC/test_latlon.csv")
df_syn_FS_LSTMTraj_best = pd.read_csv(
    r"../gen_data/LSTM-TrajGAN/syn_traj_test_weekly_FS-NYC_2000.csv"
)
df_syn_FS_exGAN_best = pd.read_csv(
    r"../gen_data/exGAN/syn_traj_test_4_weekly_FS-NYC_2000.csv"
)


df_syn_FS_TrajGDM_best_20 = pd.read_csv(
    r"..\gen_data\TrajGDM\weekly_FSNYC\syn_traj_test_l20_e397_new.csv"
)

df_original_FS = uniform_datetime_df(df_original_FS, hour_col="hour", day_col="day")
df_syn_FS_LSTMTraj_best = uniform_datetime_df(
    df_syn_FS_LSTMTraj_best, hour_col="hour", day_col="day"
)
df_syn_FS_exGAN_best = uniform_datetime_df(
    df_syn_FS_exGAN_best, hour_col="hour", day_col="day"
)

In [11]:
orig_FS = TrajectoryDataset(
    df=df_original_FS,
    dataset_name="original",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    semantic_category_col="category",
)
LSTM_Traj_FS = TrajectoryDataset(
    df=df_syn_FS_LSTMTraj_best,
    dataset_name="LSTM-TrajGAN",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    semantic_category_col="category",
)

exGAN_FS = TrajectoryDataset(
    df=df_syn_FS_exGAN_best,
    dataset_name="exGAN",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    semantic_category_col="category",
)


TrajGDM20_FS = TrajectoryDataset(
    df=df_syn_FS_TrajGDM_best_20,
    dataset_name="TrajGDM_20",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
)
data_name = "daily_geolife"

synthetics_FS = [orig_FS, LSTM_Traj_FS, exGAN_FS, TrajGDM20_FS]

In [ ]:
transition_errors = {syn.dataset_name: [] for syn in synthetics_FS}
I_errors = {syn.dataset_name: [] for syn in synthetics_FS}
s = {syn.dataset_name: [] for syn in synthetics_FS}
global_flow_errors = {syn.dataset_name: [] for syn in synthetics_FS}


def run_evaluation_step_A(gs, dx, dy, orig_FS, synthetics_FS, data_name):
    params = EvalParams(cell_width_m=gs, cell_height_m=gs, global_origin=(dx, dy))
    suite = EvaluationSuite(
        original=orig_FS,
        synthetics=synthetics_FS,
        params=params,
        data_name=data_name,
        verbose=False,  # Set to False to avoid stdout cluttering
    )

    # Compute metrics
    I_err = suite.evaluate_I_rank()
    trans_err = suite.evaluate_transtion_matrix(normalize_0_1=True)
    flow_err = [
        res["Sequential_Flow_WD"]
        for res in suite.evaluate_global_flow_prediction(normalize_0_1=True)
    ]

    # Return results mapped to synthetic names
    results = {}
    for i, syn in enumerate(synthetics_FS):
        results[syn.dataset_name] = {
            "I": I_err[i],
            "trans": trans_err[i],
            "flow": flow_err[i],
        }
    return results


gs_values = range(100, 1001, 50)
tasks = []
for gs in gs_values:
    offsets = np.linspace(0, gs, 5, endpoint=False)
    for dx in offsets:
        for dy in offsets:
            tasks.append((gs, dx, dy))


results = Parallel(n_jobs=-1)(
    delayed(run_evaluation_step_A)(gs, dx, dy, orig_FS, synthetics_FS, "Weekly FS-NYC")
    for gs, dx, dy in tqdm(tasks)
)

for i, syn in enumerate(synthetics_FS):
    syn_name = syn.dataset_name
    for res in results:
        print(res[syn_name]["I"])
        I_errors[syn_name].append(res[syn_name]["I"])
        transition_errors[syn_name].append(res[syn_name]["trans"])
        global_flow_errors[syn_name].append(res[syn_name]["flow"])

In [13]:
df_results_traj = pd.DataFrame()

df_results_traj["I-rank (WD)"] = [
    f"{np.array(I_errors[syn.dataset_name]).mean()}±{np.array(I_errors[syn.dataset_name]).std()}"
    for syn in synthetics_FS
]


header_traj = pd.DataFrame(
    [
        [f"Trajectory level ({data_name} (mean, std))"]
        + [""] * (df_results_traj.shape[1] - 1)
    ],
    columns=df_results_traj.columns,
)
df_results_traj.index = [d.dataset_name for d in synthetics_FS]
df_results_traj_FS = pd.concat([header_traj, df_results_traj])
display(df_results_traj_FS)
# # ***************** Point-level metrics ********


df_results_point = pd.DataFrame()


df_results_point["transition prob (WD)"] = [
    f"{np.array(transition_errors[syn.dataset_name]).mean()}±{np.array(transition_errors[syn.dataset_name]).std()}"
    for syn in synthetics_FS
]

df_results_point["global flow pred (W1)"] = [
    f"{np.array(global_flow_errors[syn.dataset_name]).mean()}±{np.array(global_flow_errors[syn.dataset_name]).std()}"
    for syn in synthetics_FS
]

df_results_traj.index = [d.dataset_name for d in synthetics_FS]

header_point = pd.DataFrame(
    [[f"Point-level ({data_name})"] + [""] * (df_results_point.shape[1] - 1)],
    columns=df_results_point.columns,
)

df_results_point.index = [d.dataset_name for d in synthetics_FS]
df_results_point_FS = pd.concat([header_point, df_results_point])
display(df_results_point_FS)

,I-rank (WD)
0,"Trajectory level (daily_geolife (mean, std))"
original,0.0±0.0
LSTM-TrajGAN,0.1018098305634822±0.029665590271847137
exGAN,0.13266060314139508±0.02550956743198652
TrajGDM_20,0.023686142922427118±0.01087993525039421


,transition prob (WD),global flow pred (W1)
0,Point-level (daily_geolife),
original,0.0±0.0,0.015378820963625623±0.000637547297145132
LSTM-TrajGAN,0.22999693132107663±0.14028645668226067,0.027634039565286424±0.0020025869339009707
exGAN,0.21462323503623482±0.14984830022854598,0.02934322666388939±0.003236171912387845
TrajGDM_20,0.592679924268473±0.2366476353559839,0.05133753318187882±0.007557020367953993


### Case B

In [5]:
import numpy as np
from tqdm import tqdm

from syntraj_utileval.datasets.dataset import TrajectoryDataset

from syntraj_utileval.suite.suite import (
    EvaluationSuite,
    EvalParams,
)
import pandas as pd
from joblib import Parallel, delayed

def uniform_datetime_df(df, hour_col=None, day_col=None):
    if hour_col is not None and day_col is not None:
        df["datetime"] = (
            pd.to_datetime("2024-01-01")
            + pd.to_timedelta(df[day_col], unit="D")
            + pd.to_timedelta(df[hour_col], unit="h")
        )
        df = df.drop(columns=[hour_col, day_col])
        return df

In [ ]:
df_original_Dgeo = pd.read_csv(r"..\Data\daily_geolife\test_latlon.csv")

df_original_Dgeo = uniform_datetime_df(df_original_Dgeo, hour_col="hour", day_col="day")


df_syn_Dgeo_LSTMTraj_best = pd.read_csv(
    r"../gen_data/LSTM-TrajGAN/syn_traj_test_daily_geolife_3000.csv"
)

df_syn_Dgeo_LSTMTraj_best = uniform_datetime_df(
    df_syn_Dgeo_LSTMTraj_best, hour_col="hour", day_col="day"
)

df_syn_Dgeo_exGAN_best = pd.read_csv(
    r"../gen_data/exGAN/syn_traj_test_4_daily_geolife_3000.csv"
)
df_syn_Dgeo_exGAN_best = uniform_datetime_df(
    df_syn_Dgeo_exGAN_best, hour_col="hour", day_col="day"
)

df_syn_Dgeo_TrajGDM_best_20 = pd.read_csv(
    r"..\gen_data\TrajGDM\\daily_geolife\syn_traj_test_l20_e226.csv"
)

orig_Dgeo = TrajectoryDataset(
    df=df_original_Dgeo,
    dataset_name="original",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    semantic_category_col="category",
)
LSTM_Traj_Dgeo = TrajectoryDataset(
    df=df_syn_Dgeo_LSTMTraj_best,
    dataset_name="LSTM-TrajGAN",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    semantic_category_col="category",
)

exGAN_Dgeo = TrajectoryDataset(
    df=df_syn_Dgeo_exGAN_best,
    dataset_name="exGAN",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
    semantic_category_col="category",
)


TrajGDM20_Dgeo = TrajectoryDataset(
    df=df_syn_Dgeo_TrajGDM_best_20,
    dataset_name="TrajGDM_20",
    lon_col="lon",
    lat_col="lat",
    trajectoryid_col="tid",
)
data_name = "daily_geolife"
synthetics_Dgeo = [orig_Dgeo, LSTM_Traj_Dgeo, exGAN_Dgeo, TrajGDM20_Dgeo]

In [9]:
transition_errors = {syn.dataset_name: [] for syn in synthetics_Dgeo}
G_errors = {syn.dataset_name: [] for syn in synthetics_Dgeo}
s = {syn.dataset_name: [] for syn in synthetics_Dgeo}
global_flow_errors = {syn.dataset_name: [] for syn in synthetics_Dgeo}
next_loc_errors = {syn.dataset_name: [] for syn in synthetics_Dgeo}

def run_evaluation_step_B(gs, dx, dy, orig_Dgeo, synthetics_Dgeo, data_name):
    params = EvalParams(cell_width_m=gs, cell_height_m=gs, global_origin=(dx, dy))
    suite = EvaluationSuite(
        original=orig_Dgeo,
        synthetics=synthetics_Dgeo,
        params=params,
        data_name=data_name,
        verbose=False,  # Set to False to avoid stdout cluttering
    )

    # Compute metrics
    G_err = suite.evaluate_G_rank(n_cat=10)
    trans_err = suite.evaluate_transtion_matrix(normalize_0_1=True)
    next_loc_err = [res["mean topk accuracy"]
                    for res in suite.evaluate_trajectory_next_location_prediction(
                        k=10, normalize_0_1=True
                    )]

    flow_err = [
        res["Sequential_Flow_WD"]
        for res in suite.evaluate_global_flow_prediction(normalize_0_1=True)
    ]

    # Return results mapped to synthetic names
    results = {}
    for i, syn in enumerate(synthetics_Dgeo):
        results[syn.dataset_name] = {
            "G": G_err[i],
            "trans": trans_err[i],
            "next_loc": next_loc_err[i],
            "flow": flow_err[i],
        }
    return results


gs_values = range(100, 1001, 50)
tasks = []
for gs in gs_values:
    offsets = np.linspace(0, gs, 5, endpoint=False)
    for dx in offsets:
        for dy in offsets:
            tasks.append((gs, dx, dy))


results_Dgeo = Parallel(n_jobs=-1)(
    delayed(run_evaluation_step_B)(gs, dx, dy, orig_Dgeo, synthetics_Dgeo, "Weekly FS-NYC")
    for gs, dx, dy in tqdm(tasks)
)

for i, syn in enumerate(synthetics_Dgeo):
    syn_name = syn.dataset_name
    for res in results_Dgeo:
        G_errors[syn_name].append(res[syn_name]["G"])
        transition_errors[syn_name].append(res[syn_name]["trans"])
        next_loc_errors[syn_name].append(res[syn_name]["next_loc"])
        global_flow_errors[syn_name].append(res[syn_name]["flow"])



























































100%|██████████| 475/475 [2:29:23<00:00, 18.87s/it]


In [12]:
# ******* Trajectory-level metrics ********
df_results_traj = pd.DataFrame()


df_results_traj["next loc pred (mean accuracy)"] = [
    f"{np.array(next_loc_errors[syn.dataset_name]).mean()}±{np.array(next_loc_errors[syn.dataset_name]).std()}"
    for syn in synthetics_Dgeo
]

header_traj = pd.DataFrame(
    [
        [f"Trajectory level ({data_name} (mean, std))"]
        + [""] * (df_results_traj.shape[1] - 1)
    ],
    columns=df_results_traj.columns,
)
df_results_traj.index = [d.dataset_name for d in synthetics_Dgeo]
df_results_traj_FS = pd.concat([header_traj, df_results_traj])
display(df_results_traj_FS.round(4))
# # ***************** Point-level metrics ********


df_results_point = pd.DataFrame()

df_results_point["G-rank (tau)"] = [
    f"{np.array(G_errors[syn.dataset_name]).mean()}±{np.array(G_errors[syn.dataset_name]).std()}"
    for syn in synthetics_Dgeo
]


df_results_point["transition prob (WD)"] = [
    f"{np.array(transition_errors[syn.dataset_name]).mean()}±{np.array(transition_errors[syn.dataset_name]).std()}"
    for syn in synthetics_Dgeo
]


df_results_point["global flow pred (wd)"] = [
    f"{np.array(global_flow_errors[syn.dataset_name]).mean()}±{np.array(global_flow_errors[syn.dataset_name]).std()}"
    for syn in synthetics_Dgeo
]

df_results_traj.index = [d.dataset_name for d in synthetics_Dgeo]

header_point = pd.DataFrame(
    [[f"Point-level ({data_name})"] + [""] * (df_results_point.shape[1] - 1)],
    columns=df_results_point.columns,
)

df_results_point.index = [d.dataset_name for d in synthetics_Dgeo]
df_results_point_FS = pd.concat([header_point, df_results_point])
display(df_results_point_FS.round(4))

,next loc pred (mean accuracy)
0,"Trajectory level (daily_geolife (mean, std))"
original,0.6471657188636375±0.04398981974682798
LSTM-TrajGAN,0.2755287469441259±0.15725340363190227
exGAN,0.2851333073068763±0.1591910823313206
TrajGDM_20,0.16609099580471173±0.1262439766739658


,G-rank (tau),transition prob (WD),global flow pred (wd)
0,Point-level (daily_geolife),,
original,1.0±8.99789087912733e-17,0.0±0.0,0.0081300199416401±9.759174432033979e-05
LSTM-TrajGAN,0.05120919357504683±0.18624638622172227,0.15436682345449387±0.11813643475297256,0.012149004758550356±0.0008448669676656182
exGAN,0.042175408678406585±0.18381050168122803,0.1646145695853308±0.1226919483939366,0.012193339355451057±0.0008120944254332035
TrajGDM_20,-0.1754519673260667±0.18922185318644183,0.46948052826824066±0.2796362494466732,0.025276716390832673±0.005404858054674368
